In [ ]:
# 1. Installa swig, una dipendenza necessaria per la libreria
!apt-get update
!apt-get install -y swig

# 2. Clona il repository ufficiale
!git clone https://github.com/yvan674/obb_anns.git
%cd obb_anns

# 3. Esegui lo script di setup in modalità develop
!python setup.py develop

# 4. Torna alla directory di lavoro principale del notebook
%cd ..

In [ ]:
import os
import json
from collections import Counter

# Percorsi
BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'

# Carica i dati
print("📂 Caricamento dataset...")
with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
    train_data = json.load(f)

print("\n" + "="*60)
print("🔍 ANALISI DELLA STRUTTURA DEL DATASET")
print("="*60)

# 1. Tipo e chiavi principali
print(f"\n1️⃣ Tipo di train_data: {type(train_data)}")
print(f"   Chiavi principali: {list(train_data.keys())}")

# 2. Analisi di 'categories'
categories = train_data.get('categories', {})
print(f"\n2️⃣ Categorie:")
print(f"   Tipo: {type(categories)}")
print(f"   Numero di categorie: {len(categories)}")

# Mostra le prime 3 categorie
if categories:
    print("\n   Prime 3 categorie:")
    for i, (cat_id, cat_info) in enumerate(list(categories.items())[:3]):
        print(f"     {cat_id}: {cat_info}")

# 3. Analisi di 'images'
images = train_data.get('images', [])
print(f"\n3️⃣ Immagini:")
print(f"   Tipo: {type(images)}")
print(f"   Numero di immagini: {len(images)}")

if images:
    print("\n   Prima immagine:")
    first_img = images[0]
    for key, value in first_img.items():
        print(f"     {key}: {type(value)} - {value if not isinstance(value, list) else f'lista con {len(value)} elementi'}")

# 4. Analisi di 'annotations'
annotations = train_data.get('annotations', {})
print(f"\n4️⃣ Annotazioni:")
print(f"   Tipo: {type(annotations)}")
print(f"   Numero di annotazioni: {len(annotations)}")

if annotations:
    # Prendi la prima annotazione
    first_ann_id = list(annotations.keys())[0]
    first_ann = annotations[first_ann_id]
    print(f"\n   Prima annotazione (ID: {first_ann_id}):")
    for key, value in first_ann.items():
        value_type = type(value)
        value_preview = value if not isinstance(value, list) else f"lista con {len(value)} elementi: {value[:3] if len(value) > 3 else value}"
        print(f"     {key}: {value_type} - {value_preview}")

# 5. Verifica specifica del problema: category_id
print(f"\n5️⃣ Analisi specifica di 'category_id' nelle annotazioni:")
category_id_types = []
category_id_values = []

for ann_id, ann in list(annotations.items())[:100]:  # Controlla prime 100
    cat_id = ann.get('category_id', ann.get('cat_id'))
    category_id_types.append(type(cat_id).__name__)
    category_id_values.append(cat_id)

type_counter = Counter(category_id_types)
print(f"   Tipi di category_id trovati: {dict(type_counter)}")

# Mostra esempi di valori problematici
print("\n   Esempi di valori category_id:")
for i, val in enumerate(category_id_values[:5]):
    print(f"     Esempio {i}: {val} (tipo: {type(val).__name__})")

# Se category_id è una lista, mostriamo la struttura
if any(isinstance(v, list) for v in category_id_values):
    print("\n   ⚠️ ATTENZIONE: category_id è una lista in alcune annotazioni!")
    print("   Esempio di lista category_id:")
    for v in category_id_values:
        if isinstance(v, list):
            print(f"     {v} (lunghezza: {len(v)})")
            break

# 6. Verifica struttura delle bounding box
print(f"\n6️⃣ Analisi bounding box:")
bbox_types = []
bbox_formats = []

for ann_id, ann in list(annotations.items())[:100]:
    for bbox_key in ['a_bbox', 'bbox']:
        if bbox_key in ann:
            bbox = ann[bbox_key]
            bbox_types.append(type(bbox).__name__)
            if isinstance(bbox, list):
                bbox_formats.append(f"{len(bbox)} elementi")
            break

print(f"   Tipi bbox: {Counter(bbox_types)}")
if bbox_formats:
    print(f"   Formati bbox: {Counter(bbox_formats)}")

# 7. Esempio di annotazione completa
print("\n7️⃣ Esempio di annotazione completa (la prima trovata):")
if annotations:
    sample_ann_id = list(annotations.keys())[0]
    sample_ann = annotations[sample_ann_id]
    print(json.dumps(sample_ann, indent=2)[:500] + "..." if len(json.dumps(sample_ann)) > 500 else json.dumps(sample_ann, indent=2))

print("\n" + "="*60)
print("✅ Analisi completata")

In [ ]:
import os
import json
import shutil
from tqdm import tqdm
from collections import Counter

# Percorsi
BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
OUTPUT_DIR = '/kaggle/working/deepscores_yolo'

# Crea directory di output
os.makedirs(os.path.join(OUTPUT_DIR, 'images'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'labels'), exist_ok=True)

# Carica i dati
print("📂 Caricamento dataset...")
with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
    train_data = json.load(f)

with open(os.path.join(BASE_DIR, "deepscores_test.json"), 'r') as f:
    test_data = json.load(f)

# 1. Crea mapping delle categorie (da ID stringa a indice progressivo)
categories = train_data.get('categories', {})
cat_mapping = {}
# Ordina per ID per consistenza
for idx, (cat_id, cat_info) in enumerate(sorted(categories.items(), key=lambda x: int(x[0]))):
    cat_mapping[cat_id] = idx  # cat_id è stringa come '135', '208'

print(f"✅ Categorie mappate: {len(cat_mapping)} classi (0..{len(cat_mapping)-1})")
print(f"   Esempio mapping: '135' -> {cat_mapping.get('135', 'N/A')}, '208' -> {cat_mapping.get('208', 'N/A')}")

# 2. Crea dizionari per accesso veloce
train_annotations = train_data.get('annotations', {})
test_annotations = test_data.get('annotations', {})

# Crea dizionario immagini per ID (converti img_id in int per consistenza)
train_images_by_id = {img['id']: img for img in train_data.get('images', [])}
test_images_by_id = {img['id']: img for img in test_data.get('images', [])}

print(f"✅ Annotazioni train: {len(train_annotations)}")
print(f"✅ Annotazioni test: {len(test_annotations)}")

# 3. Funzione per convertire bbox in formato YOLO
def convert_bbox_to_yolo(bbox, img_width, img_height):
    """
    Converte bbox in formato [x, y, width, height] (COCO)
    in formato YOLO [x_center, y_center, width, height] (normalizzati)
    """
    x, y, w, h = bbox
    x_center = (x + w/2) / img_width
    y_center = (y + h/2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    # Assicura che i valori siano entro [0,1]
    x_center = max(0.0, min(1.0, x_center))
    y_center = max(0.0, min(1.0, y_center))
    w_norm = max(0.0, min(1.0, w_norm))
    h_norm = max(0.0, min(1.0, h_norm))
    return [x_center, y_center, w_norm, h_norm]

# 4. Funzione per processare un dataset
def process_dataset(images_by_id, annotations_dict, name):
    print(f"\n📦 Processando {name.upper()}...")
    
    # Raggruppa annotazioni per img_id (converti img_id in int)
    anns_by_img = {}
    for ann_id, ann in annotations_dict.items():
        img_id = int(ann['img_id'])  # img_id è stringa, converti a int
        if img_id not in anns_by_img:
            anns_by_img[img_id] = []
        anns_by_img[img_id].append(ann)
    
    print(f"  Immagini con annotazioni: {len(anns_by_img)}")
    
    images_processed = 0
    total_annotations = 0
    categories_used = Counter()
    
    # Per ogni immagine, crea il file .txt
    for img_id, annotations_list in tqdm(anns_by_img.items(), desc=f"  {name}"):
        img = images_by_id.get(img_id)
        if img is None:
            print(f"\n  ⚠️ Immagine {img_id} non trovata nel dizionario")
            continue
        
        img_width = img['width']
        img_height = img['height']
        img_filename = img['filename']
        
        # Percorso del file di label
        label_filename = os.path.splitext(img_filename)[0] + '.txt'
        label_path = os.path.join(OUTPUT_DIR, 'labels', label_filename)
        
        # Scrivi le annotazioni (una riga per ogni categoria)
        with open(label_path, 'w') as f:
            for ann in annotations_list:
                # Prende la bounding box allineata (a_bbox)
                bbox = ann.get('a_bbox')
                if bbox is None:
                    continue
                
                # cat_id è una lista di stringhe, es. ['135', '208']
                cat_ids = ann.get('cat_id', [])
                if not isinstance(cat_ids, list):
                    cat_ids = [cat_ids]  # se per caso non è lista, converti
                
                # Per ogni categoria, crea una riga separata
                for cat_id in cat_ids:
                    cat_id_str = str(cat_id)
                    if cat_id_str not in cat_mapping:
                        continue
                    
                    yolo_class_id = cat_mapping[cat_id_str]
                    yolo_bbox = convert_bbox_to_yolo(bbox, img_width, img_height)
                    
                    # Scrive nel formato: class x_center y_center width height
                    f.write(f"{yolo_class_id} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")
                    total_annotations += 1
                    categories_used[yolo_class_id] += 1
        
        # Copia l'immagine
        src_img_path = os.path.join(BASE_DIR, 'images', img_filename)
        dst_img_path = os.path.join(OUTPUT_DIR, 'images', img_filename)
        
        if os.path.exists(src_img_path):
            shutil.copy2(src_img_path, dst_img_path)
            images_processed += 1
    
    print(f"  ✅ {name}: {images_processed} immagini processate")
    print(f"     Totale righe annotazioni: {total_annotations}")
    print(f"     Categorie utilizzate: {len(categories_used)}")
    
    return categories_used

# 5. Esegui la conversione
print("\n" + "="*60)
print("🚀 INIZIO CONVERSIONE")
print("="*60)

train_cats = process_dataset(train_images_by_id, train_annotations, 'train')
test_cats = process_dataset(test_images_by_id, test_annotations, 'test')

# 6. Riepilogo finale
print("\n" + "="*60)
print("📊 RIEPILOGO CONVERSIONE")
print("="*60)

images_dir = os.path.join(OUTPUT_DIR, 'images')
labels_dir = os.path.join(OUTPUT_DIR, 'labels')

print(f"  Immagini totali: {len(os.listdir(images_dir))}")
print(f"  Label file totali: {len(os.listdir(labels_dir))}")

# 7. Crea file YAML per YOLO
# Crea lista nomi delle categorie in ordine di indice
category_names = {}
for cat_id, cat_info in sorted(categories.items(), key=lambda x: int(x[0])):
    idx = cat_mapping[cat_id]
    category_names[idx] = cat_info.get('name', f'class_{cat_id}')

# Crea YAML
yaml_content = f"""# DeepScores Dense Dataset (con duplicazione annotazioni multi-categoria)
# Ogni annotazione originale con 2 categorie genera 2 righe nel file .txt

path: {OUTPUT_DIR}
train: images
val: images

nc: {len(cat_mapping)}
names: [{', '.join([f"'{category_names[i]}'" for i in range(len(cat_mapping))])}]
"""

yaml_path = '/kaggle/working/deepscores_dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"\n✅ File YAML creato: {yaml_path}")

# 8. Verifica rapida
print("\n🔍 Verifica rapida (prime 3 righe di un label file):")
sample_labels = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
if sample_labels:
    sample_path = os.path.join(labels_dir, sample_labels[0])
    print(f"  File: {sample_labels[0]}")
    with open(sample_path, 'r') as f:
        lines = f.readlines()[:3]
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id, xc, yc, w, h = parts
                print(f"    classe {class_id} ({category_names.get(int(class_id), '?')}) -> centro=({xc},{yc}) size=({w},{h})")

### CHECK BOX

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt

# Percorsi versione dense
BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'
IMG_PATH = f'{BASE_PATH}/images'

# Carica JSON
with open(JSON_FILE, 'r') as f:
    data = json.load(f)

# Prendi prima immagine
img_info = data['images'][0]
img_file = img_info['filename']
img_w = img_info['width']
img_h = img_info['height']

# Prendi PRIMA annotazione
first_ann_id = img_info['ann_ids'][0]
ann = data['annotations'][first_ann_id]

print(f"📸 Immagine: {img_file}")
print(f"📏 Dimensioni: {img_w} x {img_h}")
print(f"\n📦 PRIMA ANNOTAZIONE (ID={first_ann_id}):")
print(f"   {ann}")

# Carica immagine
img = cv2.imread(f'{IMG_PATH}/{img_file}')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Mostra immagine
plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)

# Prende bbox
bbox = ann.get('a_bbox') or ann.get('o_bbox')
if bbox:
    x, y, w, h = bbox
    print(f"\n📦 Bounding box: x={x}, y={y}, w={w}, h={h}")
    
    # Disegna solo questa bbox
    rect = plt.Rectangle((x, y), w, h, linewidth=3, edgecolor='red', facecolor='none')
    plt.gca().add_patch(rect)
    
    # Testo classe
    cat_id = ann.get('cat_id', '?')
    if isinstance(cat_id, list):
        cat_id = cat_id[0]
    plt.text(x, y-10, f'Class {cat_id}', color='red', fontsize=12, weight='bold')
    
    # Zoom intorno alla bbox
    margin = 0
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(img_w, x + w + margin)
    y2 = min(img_h, y + h + margin)
    plt.xlim(x1, x2)
    plt.ylim(y2, y1)

plt.title(f'{img_file} - SOLO prima annotazione')
plt.axis('off')
plt.show()

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt

BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'
IMG_PATH = f'{BASE_PATH}/images'

with open(JSON_FILE, 'r') as f:
    data = json.load(f)

def verifica_bbox(ann, img_w, img_h):
    """Verifica se una bbox è nel formato corretto"""
    bbox = ann.get('a_bbox')
    if not bbox:
        return "NO_BBOX"
    
    if len(bbox) != 4:
        return f"LUNGHEZZA_ERRATA:{len(bbox)}"
    
    x, y, w, h = bbox
    
    # Controlla tipo
    if isinstance(x, float) and x < 1 and isinstance(w, float) and w < 1:
        return "NORMALIZZATA"  # Probabilmente già in [0,1]
    
    if isinstance(x, (int, float)) and isinstance(w, (int, float)):
        # Controlla se è [x,y,w,h] o [x1,y1,x2,y2]
        if w > 10 and x + w < img_w + 10:  # formato [x,y,w,h]
            return "FORMATO_CORRETTO"
        elif w > 10 and w > img_w:  # potrebbe essere [x1,y1,x2,y2]
            return "FORSE_X1Y1X2Y2"
    
    return "FORMATO_INDEFINITO"

print("="*60)
print("🔍 VERIFICA FORMATO BB0X SU 20 ANNOTAZIONI CASUALI")
print("="*60)

test_images = data['images'][:5]
problemi = []

for img_info in test_images:
    img_w = img_info['width']
    img_h = img_info['height']
    img_file = img_info['filename']
    
    print(f"\n📸 {img_file} ({img_w}x{img_h})")
    
    for ann_id in img_info['ann_ids'][:3]:  # prime 3 per immagine
        ann = data['annotations'][ann_id]
        risultato = verifica_bbox(ann, img_w, img_h)
        
        bbox = ann.get('a_bbox', 'N/A')
        print(f"   ID={ann_id}: {risultato} → {bbox}")
        
        if risultato != "FORMATO_CORRETTO":
            problemi.append((img_file, ann_id, risultato, bbox))

print("\n" + "="*60)
print("📊 RIEPILOGO PROBLEMI")
print("="*60)

if problemi:
    print(f"⚠️ Trovati {len(problemi)} potenziali problemi:")
    for p in problemi[:10]:
        print(f"   {p[0]} - anno{p[1]}: {p[2]} → {p[3]}")
else:
    print("✅ Nessun problema rilevato - formato [x,y,w,h] confermato!")

In [ ]:
import os
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

# Percorsi
BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'

# Carica i dati
with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
    train_data = json.load(f)

# Prendi un'immagine e le sue annotazioni
images = train_data.get('images', [])
sample_img = images[0]  # Prima immagine
img_id = sample_img['id']

print(f"📸 Immagine: {sample_img['filename']}")
print(f"   Dimensioni dichiarate: {sample_img['width']} x {sample_img['height']}")
print()

# Carica l'immagine reale per vedere le dimensioni effettive
img_path = os.path.join(BASE_DIR, 'images', sample_img['filename'])
img_real = cv2.imread(img_path)
if img_real is not None:
    h, w = img_real.shape[:2]
    print(f"   Dimensioni reali immagine: {w} x {h}")
    print(f"   Dimensioni dichiarate vs reali: {sample_img['width']==w}, {sample_img['height']==h}")
print()

# Trova le annotazioni per questa immagine
img_annotations = []
for ann_id, ann in train_data['annotations'].items():
    if int(ann['img_id']) == img_id:
        img_annotations.append(ann)

print(f"Numero annotazioni: {len(img_annotations)}")
print()

# Analizza le prime 5 annotazioni
print("="*70)
print("ANALISI PRIME 5 ANNOTAZIONI")
print("="*70)

for i, ann in enumerate(img_annotations[:5]):
    bbox = ann.get('a_bbox')
    if bbox:
        x1, y1, x2, y2 = bbox
        print(f"\nAnnotazione {i+1}:")
        print(f"  a_bbox: [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]")
        
        # Calcola larghezza e altezza
        w = abs(x2 - x1)
        h = abs(y2 - y1)
        print(f"  Larghezza calcolata: {w:.1f}")
        print(f"  Altezza calcolata: {h:.1f}")
        print(f"  Area: {w*h:.1f}")
        
        # Verifica se le coordinate sono dentro l'immagine
        img_w, img_h = sample_img['width'], sample_img['height']
        print(f"  Dentro immagine ({img_w}x{img_h}): x1={x1>=0}, y1={y1>=0}, x2={x2<=img_w}, y2={y2<=img_h}")

# Analisi statistica
print("\n" + "="*70)
print("ANALISI STATISTICA SU 100 ANNOTAZIONI")
print("="*70)

x1_vals, y1_vals, x2_vals, y2_vals = [], [], [], []
for ann in img_annotations[:100]:
    bbox = ann.get('a_bbox')
    if bbox:
        x1, y1, x2, y2 = bbox
        x1_vals.append(x1)
        y1_vals.append(y1)
        x2_vals.append(x2)
        y2_vals.append(y2)

print(f"x1: min={min(x1_vals):.1f}, max={max(x1_vals):.1f}, media={np.mean(x1_vals):.1f}")
print(f"y1: min={min(y1_vals):.1f}, max={max(y1_vals):.1f}, media={np.mean(y1_vals):.1f}")
print(f"x2: min={min(x2_vals):.1f}, max={max(x2_vals):.1f}, media={np.mean(x2_vals):.1f}")
print(f"y2: min={min(y2_vals):.1f}, max={max(y2_vals):.1f}, media={np.mean(y2_vals):.1f}")

# Visualizza una singola annotazione per capire
print("\n" + "="*70)
print("VISUALIZZAZIONE SINGOLA ANNOTAZIONE")
print("="*70)

# Seleziona una annotazione
ann = img_annotations[0]
bbox = ann.get('a_bbox')
x1, y1, x2, y2 = bbox

# Carica l'immagine
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(15, 8))

# Immagine completa
axes[0].imshow(img)
# Disegna rettangolo
rect = plt.Rectangle((min(x1,x2), min(y1,y2)), abs(x2-x1), abs(y2-y1), 
                     linewidth=2, edgecolor='red', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title(f'Bounding box: [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]')
axes[0].axis('off')

# Zoom sull'area della bounding box
x_center = (x1 + x2) / 2
y_center = (y1 + y2) / 2
zoom_w = abs(x2 - x1) * 1.5
zoom_h = abs(y2 - y1) * 1.5

x_min = max(0, x_center - zoom_w/2)
x_max = min(img.shape[1], x_center + zoom_w/2)
y_min = max(0, y_center - zoom_h/2)
y_max = min(img.shape[0], y_center + zoom_h/2)

img_zoom = img[int(y_min):int(y_max), int(x_min):int(x_max)]
axes[1].imshow(img_zoom)
# Ricalcola rettangolo nello zoom
rect_zoom = plt.Rectangle((max(0, x1 - x_min), max(0, y1 - y_min)), 
                          abs(x2-x1), abs(y2-y1), 
                          linewidth=2, edgecolor='red', facecolor='none')
axes[1].add_patch(rect_zoom)
axes[1].set_title(f'Zoom sulla bounding box\nCategoria: {ann.get("cat_id")}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Stampa info categoria
cat_ids = ann.get('cat_id', [])
print(f"\nCategorie: {cat_ids}")
for c in cat_ids:
    if c is not None:
        cat_id_int = int(c)
        cat_name = train_data['categories'].get(str(cat_id_int), {}).get('name', 'unknown')
        print(f"  - ID {cat_id_int}: {cat_name}")

In [ ]:
import os
import json
import cv2
import matplotlib.pyplot as plt
import random
import numpy as np


# Percorsi
BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'

# Carica i dati
with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
    train_data = json.load(f)

# Crea mapping delle categorie
categories = train_data.get('categories', {})
cat_id_to_name = {int(cat_id): cat_info['name'] for cat_id, cat_info in categories.items()}

def visualize_bbox(image_id, annotations_dict, images_list, max_annotations=100):
    """
    Visualizza le bounding box usando il formato corretto [x1, y1, x2, y2]
    """
    # Trova l'immagine
    img_info = None
    for img in images_list:
        if img['id'] == image_id:
            img_info = img
            break
    
    if img_info is None:
        print(f"Immagine con ID {image_id} non trovata")
        return None
    
    # Carica l'immagine
    img_path = os.path.join(BASE_DIR, 'images', img_info['filename'])
    img = cv2.imread(img_path)
    if img is None:
        print(f"Impossibile caricare l'immagine: {img_path}")
        return None
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Trova le annotazioni per questa immagine
    img_annotations = []
    for ann_id, ann in annotations_dict.items():
        if int(ann['img_id']) == image_id:
            img_annotations.append(ann)
    
    print(f"\n📸 {img_info['filename']}")
    print(f"   Dimensioni: {img_info['width']} x {img_info['height']}")
    print(f"   Annotazioni totali: {len(img_annotations)}")
    print(f"   Visualizzate: {min(max_annotations, len(img_annotations))}")
    
    # Crea figura
    fig, ax = plt.subplots(1, 1, figsize=(15, 12))
    ax.imshow(img)
    
    # Colori per categorie
    random.seed(422)
    colors = {}
    
    for ann in img_annotations[:max_annotations]:
        bbox = ann.get('a_bbox')
        if bbox is None or len(bbox) != 4:
            continue
        
        x1, y1, x2, y2 = bbox
        
        # Assicura coordinate corrette
        x_min, x_max = min(x1, x2), max(x1, x2)
        y_min, y_max = min(y1, y2), max(y1, y2)
        w = x_max - x_min
        h = y_max - y_min
        
        # Ottieni categoria
        cat_ids = ann.get('cat_id', [])
        if cat_ids and cat_ids[0] is not None:
            cat_id = int(cat_ids[0])
            if cat_id not in colors:
                colors[cat_id] = (random.random(), random.random(), random.random())
            color = colors[cat_id]
            
            # Disegna rettangolo
            rect = plt.Rectangle((x_min, y_min), w, h, 
                                linewidth=1.5, 
                                edgecolor=color, 
                                facecolor='none')
            ax.add_patch(rect)
            
            # Etichetta
            cat_name = cat_id_to_name.get(cat_id, f'c{cat_id}')
            ax.text(x_min, y_min-3, cat_name, fontsize=7, color=color,
                   bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.7))
    
    ax.set_title(f'{img_info["filename"]} - {len(img_annotations)} annotazioni totali')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    return fig

# Test con alcune immagini
images = train_data.get('images', [])

# Mostra prime 3 immagini
for i in range(min(3, len(images))):
    print(f"\n{'='*60}")
    print(f"IMMAGINE {i+1}")
    print('='*60)
    visualize_bbox(images[i]['id'], train_data['annotations'], images)

In [ ]:
import os
import yaml

# Percorsi
OUTPUT_DIR = '/kaggle/working/deepscores_yolo'
YAML_PATH = '/kaggle/working/deepscores_dataset.yaml'

# Verifica che la directory esista
if not os.path.exists(OUTPUT_DIR):
    print(f"❌ Directory non trovata: {OUTPUT_DIR}")
    print("   Devi prima eseguire lo script di conversione!")
else:
    print(f"✅ Directory trovata: {OUTPUT_DIR}")
    
    # Conta immagini e label
    images_dir = os.path.join(OUTPUT_DIR, 'images')
    labels_dir = os.path.join(OUTPUT_DIR, 'labels')
    
    num_images = len(os.listdir(images_dir)) if os.path.exists(images_dir) else 0
    num_labels = len(os.listdir(labels_dir)) if os.path.exists(labels_dir) else 0
    
    print(f"   Immagini: {num_images}")
    print(f"   Label: {num_labels}")
    
    # Carica il dataset originale per ottenere il numero di classi
    BASE_DIR = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
    with open(os.path.join(BASE_DIR, "deepscores_train.json"), 'r') as f:
        train_data = json.load(f)
    
    categories = train_data.get('categories', {})
    num_classes = len(categories)
    
    # Crea lista nomi classi (ordine per ID)
    class_names = []
    for i in range(num_classes):
        class_names.append(f"class_{i}")
    
    # Crea YAML
    yaml_content = {
        'path': OUTPUT_DIR,
        'train': 'images',
        'val': 'images',
        'nc': num_classes,
        'names': class_names
    }
    
    # Salva YAML
    with open(YAML_PATH, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    
    print(f"\n✅ File YAML creato: {YAML_PATH}")
    print("\nContenuto:")
    print(f"  path: {OUTPUT_DIR}")
    print(f"  train: images")
    print(f"  val: images")
    print(f"  nc: {num_classes}")
    print(f"  names: [{num_classes} classi]")

In [ ]:
from ultralytics import YOLO
import optuna

def objective(trial):
    # Suggerisci iperparametri (in modo intelligente)
    lr0 = trial.suggest_float("lr0", 0.0005, 0.002, log=True)
    box = trial.suggest_float("box", 6.0, 9.0)
    cls = trial.suggest_float("cls", 0.8, 1.6)
    
    # Addestra modello
    model = YOLO("yolo11n.pt")
    results = model.train(
        data='/kaggle/working/deepscores_dataset.yaml',
        epochs=30,
        optimizer='AdamW',
        batch=6,
        workers=2,
        imgsz=320,
        lr0=lr0,
        box=box,
        cls=cls,
        device=0,
        verbose=False,
    )
    
    # Restituisci metrica da massimizzare
    return results.maps[0]  # mAP50-95

# Crea studio Optuna
study = optuna.create_study(
    direction="maximize",  # vogliamo mAP più alto
    pruner=optuna.pruners.MedianPruner()  # ferma trial inutili
)

# Esegui 50 trial (meno di grid search equivalente!)
study.optimize(objective, n_trials=20)

print(f"Migliori parametri: {study.best_params}")
print(f"Miglior mAP: {study.best_value:.4f}")

## TEST POST ACCORGIMENTI IVAN

In [ ]:
from ultralytics import YOLO
import optuna
import torch

def objective(trial):
    # Suggerisci iperparametri con range più ampi
    lr0 = trial.suggest_float("lr0", 0.0006, 0.001, log=True)  # LR iniziale più basso
    lrf = trial.suggest_float("lrf", 0.005, 0.02, log=True)    # LR finale = lr0 * lrf
    box = trial.suggest_float("box", 6.0, 9.0)
    cls = trial.suggest_float("cls", 0.8, 1.6)
    momentum = trial.suggest_float("momentum", 0.85, 0.95)      # Aggiunto momentum
    warmup_epochs = trial.suggest_int("warmup_epochs", 3, 10)   # Warmup epoche
    
    # Usa yolo11m (medium) invece di nano
    model = YOLO("yolo11m.pt")
    
    results = model.train(
        data='/kaggle/working/deepscores_dataset.yaml',
        epochs=100,                    
        patience=30,                    # Early stopping più paziente
        optimizer='AdamW',
        batch=4,                        # Batch più piccolo per yolo11m
        workers=2,
        imgsz=416,                      # Aumentato per dettaglio (da 320 a 416)
        lr0=lr0,
        lrf=lrf,                        # LR finale
        momentum=momentum,
        weight_decay=0.0005,            # Default, lasciato invariato
        warmup_epochs=warmup_epochs,    # Warmup configurabile
        warmup_bias_lr=0.1,
        warmup_momentum=0.8,
        box=box,
        cls=cls,
        device=0,
        verbose=False,
        cache=False,                    # IMPORTANTE: evita di usare RAM come cache
        rect=True,
        amp=True,                       # Mixed precision per risparmiare VRAM
        exist_ok=True,
    )
    
    # Restituisci mAP50-95
    return results.maps[0]

# Monitoraggio VRAM
print("=" * 60)
print("🔍 MONITORAGGIO RISORSE")
print("=" * 60)

# Verifica GPU disponibile
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM totale: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"   VRAM libera: {torch.cuda.memory_reserved(0) / 1024**3:.1f} GB")
else:
    print("❌ GPU non disponibile")

# Crea studio Optuna
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=3,
        n_warmup_steps=10,      # Aspetta 10 epoche prima di potare
        interval_steps=5
    )
)

print("\n🚀 Avvio tuning YOLO11m con 150 epoche...")
print("⚠️ Attenzione: Ogni trial richiederà diverse ore")
print("=" * 60)

# Riduci trial a 5-8 per evitare timeout
study.optimize(objective, n_trials=5, timeout=28800)  # Timeout 8 ore

print("\n" + "=" * 60)
print("📊 RISULTATI FINALI")
print("=" * 60)
print(f"Migliori parametri: {study.best_params}")
print(f"Miglior mAP50-95: {study.best_value:.4f}")
print(f"Miglior mAP50-95 percentuale: {study.best_value*100:.2f}%")

# Salva risultati
import json
with open('/kaggle/working/best_params_optuna.json', 'w') as f:
    json.dump({
        'best_params': study.best_params,
        'best_value': study.best_value,
        'best_trial': study.best_trial.number
    }, f, indent=2)
print("✅ Parametri salvati in /kaggle/working/best_params_optuna.json")

# Mostra tutti i trial
print("\n📋 Storico trial:")
df = study.trials_dataframe()
for i, row in df.iterrows():
    if row['state'] == 'COMPLETE':
        print(f"  Trial {int(row['number'])}: mAP={row['value']:.4f} | "
              f"lr0={row['params_lr0']:.5f} | lrf={row['params_lrf']:.4f} | "
              f"box={row['params_box']:.2f} | cls={row['params_cls']:.2f}")

# TEST con DeepScoresV2

In [ ]:
from ultralytics import YOLO
import optuna
import os
import json

DATASET_PATH = '/kaggle/input/datasets/joshuastiller/deepscoresv2/ds2_complete'

print(f"✅ Dataset trovato: {DATASET_PATH}")

# Verifica tutti i file JSON disponibili
print("\n📋 File JSON disponibili:")
train_files = []
test_files = []

for i in range(13):  # 0-12
    train_file = f'{DATASET_PATH}/deepscores-complete-{i}_train.json'
    test_file = f'{DATASET_PATH}/deepscores-complete-{i}_test.json'
    
    if os.path.exists(train_file):
        train_files.append(train_file)
        print(f"  ✅ train-{i}.json")
    if os.path.exists(test_file):
        test_files.append(test_file)
        print(f"  ✅ test-{i}.json")

print(f"\nTotale: {len(train_files)} file train, {len(test_files)} file test")

# Ispeziona la struttura del primo JSON
print("\n🔍 Ispezione struttura primo JSON:")
with open(train_files[0], 'r') as f:
    sample = json.load(f)
    print(f"Chiavi principali: {sample.keys()}")
    if 'annotations' in sample and len(sample['annotations']) > 0:
        print(f"Tipo annotations[0]: {type(sample['annotations'][0])}")
        print(f"Contenuto annotations[0]: {sample['annotations'][0][:200] if isinstance(sample['annotations'][0], str) else sample['annotations'][0]}")
    if 'images' in sample and len(sample['images']) > 0:
        print(f"Tipo images[0]: {type(sample['images'][0])}")
        print(f"Contenuto images[0]: {sample['images'][0]}")

def merge_json_files(json_files, output_path):
    merged = {
        'images': [],
        'annotations': [],
        'categories': None
    }
    
    img_id_offset = 0
    ann_id_offset = 0
    
    for jf in json_files:
        with open(jf, 'r') as f:
            data = json.load(f)
        
        # Copia categorie (sono uguali per tutti)
        if merged['categories'] is None:
            merged['categories'] = data['categories']
        
        # Aggiungi immagini
        for img in data['images']:
            img_copy = img.copy() if hasattr(img, 'copy') else img
            if isinstance(img_copy, dict):
                img_copy['id'] = img_copy.get('id', 0) + img_id_offset
            merged['images'].append(img_copy)
        
        # Aggiungi annotazioni (gestisci sia dict che str)
        for ann in data['annotations']:
            if isinstance(ann, str):
                # Se è stringa, prova a parsare come JSON
                try:
                    ann_copy = json.loads(ann)
                except:
                    print(f"⚠️ Warning: annotazione non parsabile: {ann[:100]}")
                    continue
            else:
                ann_copy = ann.copy() if hasattr(ann, 'copy') else ann
            
            if isinstance(ann_copy, dict):
                ann_copy['id'] = ann_copy.get('id', 0) + ann_id_offset
                ann_copy['img_id'] = ann_copy.get('img_id', 0) + img_id_offset
            merged['annotations'].append(ann_copy)
        
        # Aggiorna offset (solo per dict validi)
        valid_images = [img for img in merged['images'] if isinstance(img, dict) and 'id' in img]
        if valid_images:
            img_id_offset = max([img['id'] for img in valid_images]) + 1
        
        valid_anns = [ann for ann in merged['annotations'] if isinstance(ann, dict) and 'id' in ann]
        if valid_anns:
            ann_id_offset = max([ann['id'] for ann in valid_anns]) + 1
    
    # Salva JSON unito
    with open(output_path, 'w') as f:
        json.dump(merged, f)
    
    print(f"✅ Creato {output_path}: {len(merged['images'])} immagini, {len(merged['annotations'])} annotazioni")
    return output_path

# Unisci train e test separatamente
print("\n🔄 Unione file train...")
train_json = merge_json_files(train_files, '/kaggle/working/deepscores_train_merged.json')

print("\n🔄 Unione file test...")
test_json = merge_json_files(test_files, '/kaggle/working/deepscores_test_merged.json')

# Carica il train unito per ottenere il numero di categorie
with open(train_json, 'r') as f:
    train_data = json.load(f)
    num_classes = len(train_data.get('categories', []))

# Crea file data.yaml per YOLO
data_yaml = f"""
path: /kaggle/working
train: deepscores_train_merged.json
val: deepscores_test_merged.json

nc: {num_classes}
names: [\"class_{i}\" for i in range({num_classes})]
"""

with open('/kaggle/working/deepscores_dataset.yaml', 'w') as f:
    f.write(data_yaml)

print(f"\n✅ Creato deepscores_dataset.yaml con {num_classes} classi")
print("✅ Dataset pronto per YOLO!")

# RIORDINIAMO!

### Script di analisi formati

In [ ]:
import json
import numpy as np
from collections import Counter

BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'

with open(JSON_FILE, 'r') as f:
    data = json.load(f)

print("="*80)
print("🔍 ANALISI FORMATO BOUNDING BOX")
print("="*80)

# Statistiche
formati = []
dimensioni = []
rapporti = []
coordinate_negative = 0
bbox_zero = 0

for img_info in data['images']:
    img_w = img_info['width']
    img_h = img_info['height']
    
    for ann_id in img_info['ann_ids']:
        ann = data['annotations'][ann_id]
        bbox = ann.get('a_bbox')
        
        if not bbox or len(bbox) != 4:
            continue
        
        x1, y1, x3, y3 = bbox
        
        # Determina il formato
        if x3 > x1 and y3 > y1:
            # Può essere [x,y,w,h] o [x1,y1,x2,y2]
            w = x3 - x1
            h = y3 - y1
            
            if w <= 0 or h <= 0:
                bbox_zero += 1
                formato = "INVALIDO"
            elif w < 100 and h < 100 and w > 0 and h > 0:
                # Formato [x,y,w,h] con valori piccoli
                formato = "COCO [x,y,w,h]"
                formati.append("COCO")
            elif w > img_w * 0.8 or x3 > img_w * 0.9:
                # Formato [x1,y1,x2,y2]
                formato = "VOC [x1,y1,x2,y2]"
                formati.append("VOC")
            else:
                formato = "INDETERMINATO"
                formati.append("INDETERMINATO")
            
            dimensioni.append(w)
            if w > 0:
                rapporti.append(h/w)
        else:
            coordinate_negative += 1
            formati.append("ERRORE")

# Statistiche
counter = Counter(formati)
total = len(formati)

print(f"\n📊 STATISTICHE TOTALI:")
print(f"   Totale annotazioni analizzate: {total}")
print(f"\n   Formati rilevati:")
for formato, count in counter.most_common():
    percentuale = count/total*100
    print(f"      {formato}: {count} ({percentuale:.1f}%)")

print(f"\n   Annotazioni invalide: {bbox_zero}")
print(f"   Coordinate negative: {coordinate_negative}")

# Analisi dimensioni per formato
print(f"\n📏 ANALISI DIMENSIONI (larghezza in pixel):")
if dimensioni:
    print(f"   Media: {np.mean(dimensioni):.1f}")
    print(f"   Mediana: {np.median(dimensioni):.1f}")
    print(f"   Min: {np.min(dimensioni):.1f}")
    print(f"   Max: {np.max(dimensioni):.1f}")
    
    # Distribuzione
    piccole = sum(1 for d in dimensioni if d < 50)
    medie = sum(1 for d in dimensioni if 50 <= d < 200)
    grandi = sum(1 for d in dimensioni if d >= 200)
    
    print(f"\n   Distribuzione larghezze:")
    print(f"      Piccole (<50px): {piccole} ({piccole/total*100:.1f}%)")
    print(f"      Medie (50-200px): {medie} ({medie/total*100:.1f}%)")
    print(f"      Grandi (≥200px): {grandi} ({grandi/total*100:.1f}%)")

# Verifica coerenza
print(f"\n🎯 COERENZA:")
if counter.get("COCO [x,y,w,h]", 0) > total * 0.9:
    print("   ✅ FORMATO COERENTE: [x, y, width, height] (COCO)")
    print("   → Puoi usare la conversione standard o il formato nativo YOLO")
elif counter.get("VOC [x1,y1,x2,y2]", 0) > total * 0.9:
    print("   ✅ FORMATO COERENTE: [x1, y1, x2, y2] (Pascal VOC)")
    print("   → Devi convertire a [x, y, w, h] prima di usare YOLO")
else:
    print("   ⚠️ FORMATO MISTO: Il dataset ha formati diversi!")
    print("   → Necessaria conversione automatica che rilevi il formato")

# Mostra esempi
print(f"\n📝 ESEMPI CONCRETI:")
esempi_mostrati = 0
for img_info in data['images'][:3]:
    for ann_id in img_info['ann_ids'][:2]:
        ann = data['annotations'][ann_id]
        bbox = ann.get('a_bbox')
        if bbox and len(bbox) == 4:
            print(f"\n   Annotazione {ann_id}: {bbox}")
            print(f"   cat_id: {ann.get('cat_id')}")
            esempi_mostrati += 1
            if esempi_mostrati >= 6:
                break
    if esempi_mostrati >= 6:
        break

print("\n" + "="*80)
print("💡 CONCLUSIONE:")
if counter.get("COCO [x,y,w,h]", 0) > total * 0.95:
    print("   Usa YOLO nativamente con JSON COCO")
elif counter.get("VOC [x1,y1,x2,y2]", 0) > total * 0.95:
    print("   Converti da [x1,y1,x2,y2] a [x,y,w,h] prima di usare YOLO")
else:
    print("   Implementa conversione automatica che supporti entrambi i formati")
print("="*80)

## VOC O COCO?
DUBBIO: Il dataset DeepScoresV2Dense è MISTO alcune classi sono annotate come VOC altre come COCO? Risposta: FALSO, tutti VOC (i check successivi lo dimostrano)

### Interactive check

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'
IMG_PATH = f'{BASE_PATH}/images'

# Carica dati
with open(JSON_FILE, 'r') as f:
    data = json.load(f)

# Filtra categorie valide
categories = {}
for cat_id, cat_info in data.get('categories', {}).items():
    try:
        int(cat_id)
        categories[cat_id] = cat_info
    except (ValueError, TypeError):
        continue

print("=" * 80)
print("🔍 VALUTAZIONE MANUALE: VOC vs COCO per ogni classe")
print("=" * 80)

# Raccogli annotazioni per classe
annotazioni_per_classe = defaultdict(list)

for img_info in data['images'][:200]:  # Prime 200 immagini
    img_w = img_info['width']
    img_h = img_info['height']
    
    for ann_id in img_info.get('ann_ids', []):
        ann = data['annotations'].get(ann_id)
        if ann:
            bbox = ann.get('a_bbox')
            if bbox:
                cat_ids = ann.get('cat_id', [])
                if not isinstance(cat_ids, list):
                    cat_ids = [cat_ids]
                
                for cat_id in cat_ids:
                    cat_id_str = str(cat_id)
                    if cat_id_str in categories:
                        annotazioni_per_classe[cat_id_str].append({
                            'img_file': img_info['filename'],
                            'bbox': bbox,
                            'img_w': img_w,
                            'img_h': img_h,
                            'ann_id': ann_id
                        })

# Per ogni classe, mostra un esempio come VOC e uno come COCO
classi_da_mostrare = list(annotazioni_per_classe.keys())[:30]  # Prime 30 classi

for cat_id in classi_da_mostrare:
    cat_name = categories[cat_id].get('name', f'classe_{cat_id}')
    annotazioni = annotazioni_per_classe[cat_id]
    
    if len(annotazioni) < 2:
        continue
    
    # Prendi una annotazione
    ann = annotazioni[0]
    bbox = ann['bbox']
    img_w = ann['img_w']
    img_h = ann['img_h']
    img_file = ann['img_file']
    x1, y1, x3, y3 = bbox
    
    # Calcola come VOC e COCO
    w_voc = x3 - x1
    h_voc = y3 - y1
    
    # Calcola coordinate YOLO per entrambi i formati
    # Formato VOC
    xc_voc = (x1 + w_voc/2) / img_w
    yc_voc = (y1 + h_voc/2) / img_h
    w_voc_norm = w_voc / img_w
    h_voc_norm = h_voc / img_h
    
    # Formato COCO
    xc_coco = (x1 + x3/2) / img_w
    yc_coco = (y1 + y3/2) / img_h
    w_coco_norm = x3 / img_w
    h_coco_norm = y3 / img_h
    
    # Carica immagine
    img = cv2.imread(f'{IMG_PATH}/{img_file}')
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Crea figura
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # SINISTRA: Interpretazione VOC
    axes[0].imshow(img_rgb)
    rect_voc = plt.Rectangle((x1, y1), w_voc, h_voc, 
                             linewidth=2, edgecolor='red', facecolor='none')
    axes[0].add_patch(rect_voc)
    axes[0].set_title(f'VOC: [{x1:.0f}, {y1:.0f}, {x3:.0f}, {y3:.0f}]\n'
                      f'Larghezza={w_voc:.0f}px, Altezza={h_voc:.0f}px')
    axes[0].axis('off')
    
    # DESTRA: Interpretazione COCO
    axes[1].imshow(img_rgb)
    rect_coco = plt.Rectangle((x1, y1), x3, y3, 
                              linewidth=2, edgecolor='blue', facecolor='none')
    axes[1].add_patch(rect_coco)
    axes[1].set_title(f'COCO: [{x1:.0f}, {y1:.0f}, {x3:.0f}, {y3:.0f}]\n'
                      f'Larghezza={x3:.0f}px, Altezza={y3:.0f}px')
    axes[1].axis('off')
    
    plt.suptitle(f'Classe {cat_id}: {cat_name}', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Input manuale
    print(f"\n📋 Classe {cat_id}: {cat_name}")
    print("   Quale interpretazione è CORRETTA?")
    print("    1 = VOC (rettangolo ROSSO)")
    print("    2 = COCO (rettangolo BLU)")
    print("    0 = SALTA / nessuna delle due")
    
    risposta = input("   → Scegli (0/1/2): ")
    
    # Salva risultato
    with open('/kaggle/working/valutazione_classi.txt', 'a') as f:
        f.write(f"{cat_id},{cat_name},{risposta},{w_voc},{x3}\n")
    
    plt.close()

print("\n" + "=" * 80)
print("✅ VALUTAZIONE COMPLETATA")
print("   Risultati salvati in: /kaggle/working/valutazione_classi.txt")
print("=" * 80)

# Mostra riepilogo
print("\n📊 RIEPILOGO VALUTAZIONI:")
with open('/kaggle/working/valutazione_classi.txt', 'r') as f:
    righe = f.readlines()

votazioni = {'VOC': 0, 'COCO': 0, 'SALTA': 0}
for riga in righe:
    parts = riga.strip().split(',')
    if len(parts) >= 3:
        if parts[2] == '1':
            votazioni['VOC'] += 1
        elif parts[2] == '2':
            votazioni['COCO'] += 1
        elif parts[2] == '0':
            votazioni['SALTA'] += 1

print(f"   VOC corrette: {votazioni['VOC']}")
print(f"   COCO corrette: {votazioni['COCO']}")
print(f"   Saltate: {votazioni['SALTA']}")

### Static check

In [50]:
import json
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.backends.backend_pdf import PdfPages

BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
JSON_FILE = f'{BASE_PATH}/deepscores_train.json'
IMG_PATH = f'{BASE_PATH}/images'

print("📂 Caricamento JSON...")
with open(JSON_FILE, 'r') as f:
    data = json.load(f)

# Filtra categorie
categories = {}
for cat_id, cat_info in data.get('categories', {}).items():
    try:
        int(cat_id)
        categories[cat_id] = cat_info
    except (ValueError, TypeError):
        continue

print(f"✅ {len(categories)} classi trovate")

# Raccogli annotazioni
annotazioni_per_classe = defaultdict(list)
for img_info in data['images'][:200]:
    for ann_id in img_info.get('ann_ids', []):
        ann = data['annotations'].get(ann_id)
        if ann and ann.get('a_bbox'):
            cat_ids = ann.get('cat_id', [])
            if not isinstance(cat_ids, list):
                cat_ids = [cat_ids]
            for cat_id in cat_ids:
                cat_id_str = str(cat_id)
                if cat_id_str in categories:
                    annotazioni_per_classe[cat_id_str].append({
                        'img_file': img_info['filename'],
                        'bbox': ann['a_bbox'],
                        'img_w': img_info['width'],
                        'img_h': img_info['height']
                    })

# Crea PDF con tutte le classi
pdf_path = '/kaggle/working/verifica_classi_VOC_vs_COCO.pdf'
print(f"\n📄 Creazione PDF: {pdf_path}")

with PdfPages(pdf_path) as pdf:
    for cat_id in sorted(annotazioni_per_classe.keys(), key=lambda x: int(x)):
        annotazioni = annotazioni_per_classe[cat_id]
        cat_name = categories[cat_id].get('name', f'classe_{cat_id}')
        
        # Prendi una annotazione
        ann = annotazioni[0]
        bbox = ann['bbox']
        img_w = ann['img_w']
        img_h = ann['img_h']
        x1, y1, x3, y3 = bbox
        
        w_voc = x3 - x1
        h_voc = y3 - y1
        
        # Carica immagine
        img = cv2.imread(f'{IMG_PATH}/{ann["img_file"]}')
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Crea figura
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # VOC (ROSSO)
        axes[0].imshow(img_rgb)
        axes[0].add_patch(plt.Rectangle((x1, y1), w_voc, h_voc, 
                                        linewidth=2, edgecolor='red', facecolor='none'))
        axes[0].set_title(f'VOC: [{x1:.0f}, {y1:.0f}, {x3:.0f}, {y3:.0f}]\nlarghezza={w_voc:.0f}px', fontsize=10)
        axes[0].axis('off')
        
        # COCO (BLU)
        axes[1].imshow(img_rgb)
        axes[1].add_patch(plt.Rectangle((x1, y1), x3, y3, 
                                        linewidth=2, edgecolor='blue', facecolor='none'))
        axes[1].set_title(f'COCO: [{x1:.0f}, {y1:.0f}, {x3:.0f}, {y3:.0f}]\nlarghezza={x3:.0f}px', fontsize=10)
        axes[1].axis('off')
        
        fig.suptitle(f'Classe {cat_id}: {cat_name}', fontsize=12)
        plt.tight_layout()
        
        pdf.savefig(fig)
        plt.close(fig)
        
        # Progresso
        print(f"  Aggiunta classe {cat_id}: {cat_name}")

print("\n" + "=" * 60)
print(f"✅ PDF CREATO: {pdf_path}")
print("   Scarica il file e osserva ogni coppia:")
print("   - ROSSO = interpretazione VOC [x1,y1,x2,y2]")
print("   - BLU = interpretazione COCO [x,y,w,h]")
print("   - Il rettangolo CORRETTO deve circondare esattamente l'oggetto")
print("=" * 60)

# Versione alternativa: stampa tabella riassuntiva
print("\n📊 TABELLA RIASSUNTIVA (prime 50 classi):")
print("-" * 80)
print(f"{'Classe':<8} {'Nome':<30} {'w_voc':<8} {'w_coco':<8} {'Formato suggerito':<15}")
print("-" * 80)

for cat_id in sorted(annotazioni_per_classe.keys(), key=lambda x: int(x))[:50]:
    annotazioni = annotazioni_per_classe[cat_id]
    cat_name = categories[cat_id].get('name', f'classe_{cat_id}')[:28]
    ann = annotazioni[0]
    bbox = ann['bbox']
    x1, y1, x3, y3 = bbox
    w_voc = x3 - x1
    w_coco = x3
    
    img_w = ann['img_w']
    soglia_relativa = img_w * 0.10
    
    if w_voc > 0 and w_voc < soglia_relativa:
        suggerito = "VOC"
    else:
        suggerito = "COCO"
    
    print(f"{cat_id:<8} {cat_name:<30} {w_voc:<8.0f} {w_coco:<8.0f} {suggerito:<15}")

print("-" * 80)
print("\n💡 VALUTA CON I TUOI OCCHI: apri il PDF e controlla")
print("   Se il rettangolo suggerito NON circonda correttamente l'oggetto,")
print("   allora la soglia del 10% va modificata per quella classe.")

📂 Caricamento JSON...
✅ 208 classi trovate

📄 Creazione PDF: /kaggle/working/verifica_classi_VOC_vs_COCO.pdf
  Aggiunta classe 1: brace
  Aggiunta classe 2: ledgerLine
  Aggiunta classe 3: repeatDot
  Aggiunta classe 4: segno
  Aggiunta classe 5: coda
  Aggiunta classe 6: clefG
  Aggiunta classe 7: clefCAlto
  Aggiunta classe 8: clefCTenor
  Aggiunta classe 9: clefF
  Aggiunta classe 11: clef8
  Aggiunta classe 12: clef15
  Aggiunta classe 13: timeSig0
  Aggiunta classe 14: timeSig1
  Aggiunta classe 15: timeSig2
  Aggiunta classe 16: timeSig3
  Aggiunta classe 17: timeSig4
  Aggiunta classe 18: timeSig5
  Aggiunta classe 19: timeSig6
  Aggiunta classe 20: timeSig7
  Aggiunta classe 21: timeSig8
  Aggiunta classe 22: timeSig9
  Aggiunta classe 23: timeSigCommon
  Aggiunta classe 24: timeSigCutCommon
  Aggiunta classe 25: noteheadBlackOnLine
  Aggiunta classe 27: noteheadBlackInSpace
  Aggiunta classe 29: noteheadHalfOnLine
  Aggiunta classe 31: noteheadHalfInSpace
  Aggiunta classe 33:

## E' stato appurato che il formato NON è misto, ma sono tutti VOC!

## Creo YAML da JSON

In [1]:
import json
import os
import shutil
from tqdm import tqdm

BASE_PATH = '/kaggle/input/datasets/antonioquattromini/deepscoredense/ds2_dense'
OUTPUT_DIR = '/kaggle/working/deepscores_yolo'

os.makedirs(os.path.join(OUTPUT_DIR, 'images'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'labels'), exist_ok=True)

print("=" * 60)
print("CONVERSIONE DeepScoresV2 → YOLO")
print("=" * 60)

# Carica dati
with open(os.path.join(BASE_PATH, "deepscores_train.json"), 'r') as f:
    train_data = json.load(f)

print(f"📂 Train: {len(train_data.get('images', []))} immagini")

# ============================================
# 1. CREA MAPPING PROGRESSIVO (ignora None)
# ============================================
print("\n🔍 Analisi classi presenti...")

classi_presenti = set()
for ann in train_data['annotations'].values():
    cat_ids = ann.get('cat_id')
    if cat_ids is None:
        continue
    if isinstance(cat_ids, list):
        for c in cat_ids:
            if c is not None:
                try:
                    classi_presenti.add(int(c))
                except (ValueError, TypeError):
                    pass
    else:
        try:
            classi_presenti.add(int(cat_ids))
        except (ValueError, TypeError):
            pass

print(f"   Classi trovate: {len(classi_presenti)}")

if len(classi_presenti) == 0:
    print("   ❌ Nessuna classe valida trovata!")
    exit()

# Crea mapping ID originale → indice progressivo
id_to_idx = {orig_id: idx for idx, orig_id in enumerate(sorted(classi_presenti))}
print(f"   Mapping: 0..{len(id_to_idx)-1}")

# ============================================
# 2. FUNZIONE CONVERSIONE
# ============================================
def voc_to_yolo(bbox, img_w, img_h):
    if len(bbox) != 4:
        return None
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    
    if w <= 0 or h <= 0:
        return None
    
    x_center = (x1 + w/2) / img_w
    y_center = (y1 + h/2) / img_h
    w_norm = w / img_w
    h_norm = h / img_h
    
    return [max(0.0, min(1.0, x_center)), 
            max(0.0, min(1.0, y_center)),
            max(0.0, min(1.0, w_norm)), 
            max(0.0, min(1.0, h_norm))]

# ============================================
# 3. PROCESSING
# ============================================
print("\n📦 Processando immagini...")

# Crea indice veloce delle annotazioni per img_id
anns_by_img = {}
for ann_id, ann in train_data['annotations'].items():
    img_id = ann.get('img_id')
    if img_id is not None:
        if img_id not in anns_by_img:
            anns_by_img[img_id] = []
        anns_by_img[img_id].append(ann)

images_processed = 0
total_annotations = 0
errori_bbox = 0
errori_classe = 0

for img in tqdm(train_data['images'], desc="   Progresso"):
    img_id = img['id']
    img_filename = img['filename']
    img_w = img['width']
    img_h = img['height']
    
    anns_for_img = anns_by_img.get(img_id, [])
    if not anns_for_img:
        continue
    
    # Crea file label
    label_name = os.path.splitext(img_filename)[0] + '.txt'
    label_path = os.path.join(OUTPUT_DIR, 'labels', label_name)
    
    with open(label_path, 'w') as f:
        for ann in anns_for_img:
            bbox = ann.get('a_bbox')
            if not bbox:
                errori_bbox += 1
                continue
            
            yolo_bbox = voc_to_yolo(bbox, img_w, img_h)
            if yolo_bbox is None:
                errori_bbox += 1
                continue
            
            cat_ids = ann.get('cat_id')
            if cat_ids is None:
                errori_classe += 1
                continue
            
            if not isinstance(cat_ids, list):
                cat_ids = [cat_ids]
            
            for cat_id in cat_ids:
                if cat_id is None:
                    continue
                try:
                    orig_id = int(cat_id)
                    if orig_id not in id_to_idx:
                        errori_classe += 1
                        continue
                    
                    class_idx = id_to_idx[orig_id]
                    f.write(f"{class_idx} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}\n")
                    total_annotations += 1
                except (ValueError, TypeError):
                    errori_classe += 1
                    continue
    
    # Copia immagine
    src = os.path.join(BASE_PATH, 'images', img_filename)
    dst = os.path.join(OUTPUT_DIR, 'images', img_filename)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        images_processed += 1

print(f"\n✅ Risultati:")
print(f"   Immagini processate: {images_processed}")
print(f"   Annotazioni scritte: {total_annotations}")
print(f"   Errori bbox: {errori_bbox}")
print(f"   Errori classe: {errori_classe}")

if total_annotations == 0:
    print("\n❌ ERRORE: Nessuna annotazione scritta!")
    exit()

# ============================================
# 4. CREA YAML
# ============================================
categories = train_data.get('categories', {})
names_list = []
for orig_id in sorted(id_to_idx.keys()):
    cat_info = categories.get(str(orig_id), {})
    name = cat_info.get('name', f'class_{orig_id}')
    names_list.append(name)

yaml_content = f"""path: {OUTPUT_DIR}
train: images
val: images

nc: {len(names_list)}
names: {names_list}
"""

with open('/kaggle/working/deepscores_dataset.yaml', 'w') as f:
    f.write(yaml_content)

print(f"\n📄 YAML creato: /kaggle/working/deepscores_dataset.yaml")
print(f"   nc: {len(names_list)}")
print(f"   Prime 5 classi: {names_list[:5]}")

CONVERSIONE DeepScoresV2 → YOLO
📂 Train: 1362 immagini

🔍 Analisi classi presenti...
   Classi trovate: 185
   Mapping: 0..184

📦 Processando immagini...


   Progresso: 100%|██████████| 1362/1362 [00:00<00:00, 996970.69it/s]


✅ Risultati:
   Immagini processate: 0
   Annotazioni scritte: 0
   Errori bbox: 0
   Errori classe: 0

❌ ERRORE: Nessuna annotazione scritta!

📄 YAML creato: /kaggle/working/deepscores_dataset.yaml
   nc: 185
   Prime 5 classi: ['brace', 'ledgerLine', 'repeatDot', 'segno', 'coda']


## ADDESTRAMENTO

In [55]:
from ultralytics import YOLO
import optuna

def objective(trial):
    # Iperparametri da ottimizzare (range basati su consigli Ivan)
    lr0 = trial.suggest_float("lr0", 0.0005, 0.001, log=True)      # LR iniziale (Ivan: 0.0005)
    lrf = trial.suggest_float("lrf", 0.005, 0.02, log=True)        # LR finale = lr0 * lrf (Ivan: 0.01)
    momentum = trial.suggest_float("momentum", 0.85, 0.95)          # Momento (default: 0.937)
    warmup_epochs = trial.suggest_int("warmup_epochs", 3, 10)       # Warmup (Ivan: 5-10)
    box = trial.suggest_float("box", 6.0, 9.0)                      # Box loss weight
    cls = trial.suggest_float("cls", 0.8, 1.6)                      # Class loss weight
    
    model = YOLO("yolo11m.pt")  # ← MEDIUM, non Nano!
    
    results = model.train(
        # DATASET (JSON originale, nessuna conversione!)
        data='/kaggle/working/deepscores_dataset.yaml',
        
        # EPOCHE (Ivan: 100-150)
        epochs=100,
        patience=30,                    # Early stopping più paziente
        
        # OTTIMIZZATORE
        optimizer='AdamW',
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=0.0005,            # Default, lasciato invariato
        
        # WARMUP (Ivan: 5-10 epoche)
        warmup_epochs=warmup_epochs,
        warmup_bias_lr=0.1,
        warmup_momentum=0.8,
        
        # LOSS WEIGHTS
        box=box,
        cls=cls,
        
        # DATALOADER
        batch=4,                        # Piccolo per yolo11m + immagini grandi
        workers=2,
        rect=True,                      # Migliora efficienza
        cache=False,                    # NON cache in RAM (importante!)
        
        # IMMAGINI
        imgsz=640,                      # Ivan: più grande possibile (640 invece di 320)
        
        # ALTRO
        device=0,
        verbose=True,
        amp=True,                       # Mixed precision (risparmia VRAM)
        exist_ok=True,
    )
    
    return results.maps[0]  # mAP50-95

# Optuna studio
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=10)
)

# Solo 5-8 trial (ogni trial richiede ore!)
study.optimize(objective, n_trials=5, timeout=28800)

print(f"Migliori parametri: {study.best_params}")
print(f"Miglior mAP: {study.best_value:.4f}")

[I 2026-06-04 14:59:58,959] A new study created in memory with name: no-name-80cc754d-69fa-4167-9493-3cbc8e9ffc80


Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=6.078628414479082, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.2455979528905274, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/deepscores_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.000795530153292275, lrf=0.0065622764908313935, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.8949819052625471, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=Fa

[W 2026-06-04 15:00:09,484] Trial 0 failed with parameters: {'lr0': 0.000795530153292275, 'lrf': 0.0065622764908313935, 'momentum': 0.8949819052625471, 'warmup_epochs': 6, 'box': 6.078628414479082, 'cls': 1.2455979528905274} because of the following error: RuntimeError('No valid images found in /kaggle/working/deepscores_yolo/labels.cache.\n  \x1b\x1btrain: \x1b/kaggle/working/deepscores_yolo/images/lg-101766503886095953-aug-beethoven--page-1.png: ignoring corrupt image/label: Label class 207 exceeds dataset class count 168. Possible class labels are 0-167\n  \x1b\x1btrain: \x1b/kaggle/working/deepscores_yolo/images/lg-101766503886095953-aug-beethoven--page-4.png: ignoring corrupt image/label: Label class 207 exceeds dataset class count 168. Possible class labels are 0-167\n  \x1b\x1btrain: \x1b/kaggle/working/deepscores_yolo/images/lg-101766503886095953-aug-emmentaler--page-4.png: ignoring corrupt image/label: Label class 207 exceeds dataset class count 168. Possible class labels are 

RuntimeError: No valid images found in /kaggle/working/deepscores_yolo/labels.cache.
  [34m[1mtrain: [0m/kaggle/working/deepscores_yolo/images/lg-101766503886095953-aug-beethoven--page-1.png: ignoring corrupt image/label: Label class 207 exceeds dataset class count 168. Possible class labels are 0-167
  [34m[1mtrain: [0m/kaggle/working/deepscores_yolo/images/lg-101766503886095953-aug-beethoven--page-4.png: ignoring corrupt image/label: Label class 207 exceeds dataset class count 168. Possible class labels are 0-167
  [34m[1mtrain: [0m/kaggle/working/deepscores_yolo/images/lg-101766503886095953-aug-emmentaler--page-4.png: ignoring corrupt image/label: Label class 207 exceeds dataset class count 168. Possible class labels are 0-167
  [34m[1mtrain: [0m/kaggle/working/deepscores_yolo/images/lg-101766503886095953-aug-gonville--page-1.png: ignoring corrupt image/label: Label class 207 exceeds dataset class count 168. Possible class labels are 0-167
  [34m[1mtrain: [0m/kaggle/w